<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_53_rust_for_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🦀 **Module 53 — Rust Crash Course (the hot-path language for AI)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 53 — Rust Crash Course

> Go (M52) is the cloud-native glue. **Rust** is what runs on the **hot path**: HuggingFace **`tokenizers`** is Rust. **`safetensors`** is Rust. **Polars** (the dataframe library that's eating Pandas) is Rust. **Qdrant** (M42) is Rust. **`candle`** (HF's pure-Rust ML framework), **`burn`**, **`mistral.rs`**, **`uv`** (the Astral pip replacement), **`ruff`**, **`tiktoken`**'s fast paths — all Rust. The pattern is consistent: when an ML tool needs to be *fast and safe*, the answer in 2026 is Rust.
>
> This module is a Rust crash course aimed at AI tooling. **No "learn Rust in 21 days"** — the 20% that lets you read those crates, contribute small fixes, and write a Rust extension for your Python code via **PyO3**.

> 🟡 We'll install Rust (`rustup`) in Colab and run real programs.

### What you'll cover
1. Why Rust shows up everywhere in ML 2026
2. Setup — install Rust + `cargo new` + hello world
3. The big idea: **ownership + borrowing** in 5 minutes
4. Syntax fast-tour: types, structs, enums, `Result<T, E>`
5. **Iterators + closures** — Rust's "Pythonic" side
6. **Cargo** — the package manager (no `pip` problems)
7. **`tokio`** + **`axum`** — async HTTP server
8. **PyO3** — write a Rust function callable from Python
9. The ML-Rust ecosystem you'll actually use (`candle`, `polars`, `tokenizers`, `safetensors`)
10. Rust vs Go vs C++ — when to pick what


## 1 · Why Rust in ML 2026

| Property | Why ML tooling cares |
|---|---|
| **No GC, no runtime** | predictable latency in inference / hot loops |
| **Memory safety** without a garbage collector | C++ speed, no segfaults / use-after-free |
| **`unsafe` is opt-in** and audited | safe by default, escape hatch where needed |
| **First-class FFI** | drop-in replacement for C extensions (PyO3, cxx) |
| **`tokio` async** | high-concurrency I/O like Go but with zero-cost abstractions |
| **Cargo** | the build/dependency story Python wishes it had |
| **`rust-cuda` + `cudarc`** | first-class CUDA bindings; some kernels written in pure Rust now |

### Where Rust has won in ML already
- **Tokenizers** (HF) — pure Rust, ~50× faster than the old Python ones.
- **Polars** — DataFrame replacement, multi-threaded by default, lazy plans.
- **Safetensors** — the "no Python pickle" model format.
- **uv / ruff / pyright-rust competitors** — Python *tooling* is being rewritten in Rust.
- **Qdrant** (M42) — vector DB, pure Rust, Apache 2 licensed.
- **`candle` / `burn` / `mistral.rs`** — Rust-native deep-learning frameworks.

### Where you should NOT pick Rust
- ❌ Anywhere a Python script of < 200 LoC will work.
- ❌ Replacing Go for k8s control loops or HTTP gateways — Go's stdlib + goroutines are usually enough and faster to hire for.
- ❌ Whole training stacks — PyTorch ecosystem (M16-M24, M39) is years ahead.


## 2 · Setup

In [ ]:
# install rustup → toolchain
!curl -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal > /dev/null 2>&1
import os
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
!rustc --version && cargo --version

In [ ]:
!cd /content && cargo new hello --quiet && cat /content/hello/src/main.rs

In [ ]:
!cd /content/hello && cargo run --quiet

## 3 · Ownership + borrowing

This is the entire reason Rust is hard *and* the entire reason it's safe. Three rules:

1. **Every value has exactly one owner.**
2. **When the owner goes out of scope, the value is freed.** (No GC needed.)
3. You can hand out **borrows** — references — as long as they obey the **borrow checker**:
   - any number of `&T` (shared, read-only) **OR**
   - exactly one `&mut T` (exclusive, read-write)
   - never both at the same time

That last bit — **shared XOR mutable** — is the entire trick. It's what eliminates whole classes of bugs (data races, iterator invalidation, use-after-free) at compile time.

In [ ]:
ownership = '''fn main() {
    let s = String::from("hello");          // s OWNS the heap-allocated bytes
    let len = take_len(&s);                 // borrow read-only — s still alive after
    println!("{:?} -> len {}", s, len);

    let mut v = vec![1, 2, 3];
    push_one(&mut v);                       // borrow mutably (exclusive)
    println!("v = {:?}", v);
}

fn take_len(s: &String) -> usize {          // &String == read-only borrow
    s.len()
}

fn push_one(v: &mut Vec<i32>) {             // &mut == mutable borrow
    v.push(99);
}
'''
import os
os.makedirs("/content/own/src", exist_ok=True)
open("/content/own/Cargo.toml","w").write(
    '[package]\nname = "own"\nversion = "0.1.0"\nedition = "2021"\n\n[dependencies]\n')
open("/content/own/src/main.rs","w").write(ownership)
!cd /content/own && cargo run --quiet 2>&1 | head -10

## 4 · Syntax fast-tour

In [ ]:
tour = '''use std::collections::HashMap;

// STRUCT — like a Go struct or Python dataclass
#[derive(Debug)]
struct Model {
    name: String,
    params: u64,
    active: bool,
}

// METHODS — `impl` block
impl Model {
    fn tag(&self) -> String {
        format!("{}-{}", self.name.to_lowercase(), self.params)
    }
}

// ENUM — sum types (algebraic). Result and Option are stdlib enums.
enum Backend { OpenAI, Anthropic, VLLM }

// Result<T, E> for error handling — no exceptions
fn divide(a: i32, b: i32) -> Result<i32, String> {
    if b == 0 { Err("divide by zero".into()) } else { Ok(a / b) }
}

fn main() {
    let m = Model { name: "Qwen".into(), params: 500_000_000, active: true };
    println!("tag = {}", m.tag());

    let mut cfg: HashMap<&str, i32> = HashMap::new();
    cfg.insert("max_tokens", 256);
    cfg.insert("top_k", 50);
    for (k, v) in &cfg { println!("{} = {}", k, v); }

    // The ? operator unwraps Ok or returns the error early
    match divide(10, 0) {
        Ok(q)  => println!("q = {}", q),
        Err(e) => println!("oops: {}", e),
    }

    let b = Backend::VLLM;
    match b {
        Backend::OpenAI    => println!("calling openai"),
        Backend::Anthropic => println!("calling anthropic"),
        Backend::VLLM      => println!("calling vllm locally"),
    }
}
'''
os.makedirs("/content/tour/src", exist_ok=True)
open("/content/tour/Cargo.toml","w").write(
    '[package]\nname = "tour"\nversion = "0.1.0"\nedition = "2021"\n\n[dependencies]\n')
open("/content/tour/src/main.rs","w").write(tour)
!cd /content/tour && cargo run --quiet 2>&1 | tail -12

**Highlights.**
- `Result<T, E>` and `Option<T>` are how Rust handles "this might fail" / "this might be missing." No exceptions, no `null`.
- The `?` operator unwraps `Ok` and **returns the error early** — the equivalent of Go's `if err != nil { return err }` in one character.
- `match` is **exhaustive** — the compiler errors if you don't cover every variant.
- `derive(Debug)` auto-generates a printable representation.

## 5 · Iterators + closures — Rust's "Pythonic" side

In [ ]:
iters = '''fn main() {
    let xs = vec![1,2,3,4,5,6,7,8];

    // map + filter + sum, all lazy, all on the stack
    let s: i32 = xs.iter()
                   .map(|x| x * x)
                   .filter(|x| x % 2 == 0)
                   .sum();
    println!("sum of even squares = {}", s);

    // collect into a Vec
    let doubled: Vec<i32> = xs.iter().map(|x| x * 2).collect();
    println!("{:?}", doubled);

    // for ... in works on anything that implements IntoIterator
    for (i, v) in xs.iter().enumerate() {
        if v % 3 == 0 { println!("idx {} -> {}", i, v); }
    }
}
'''
os.makedirs("/content/iters/src", exist_ok=True)
open("/content/iters/Cargo.toml","w").write(
    '[package]\nname = "iters"\nversion = "0.1.0"\nedition = "2021"\n\n[dependencies]\n')
open("/content/iters/src/main.rs","w").write(iters)
!cd /content/iters && cargo run --quiet 2>&1 | tail -8

Iterator chains compile to the same machine code as a hand-written `for` loop. **Zero-cost abstractions.** Combine with **`rayon`** (`use rayon::prelude::*; xs.par_iter().map(...)...`) and you have data-parallelism for free across all cores.

## 6 · Cargo — the package manager Python wishes it had

```toml
# Cargo.toml
[package]
name = "llm-gateway"
version = "0.1.0"
edition = "2021"

[dependencies]
tokio       = { version = "1", features = ["full"] }
axum        = "0.7"
serde       = { version = "1", features = ["derive"] }
serde_json  = "1"
reqwest     = { version = "0.12", features = ["json"] }

[profile.release]
lto = "fat"
codegen-units = 1
```

Then:
```bash
cargo build --release       # one static binary
cargo test                  # run the test suite
cargo bench                 # microbenchmarks
cargo clippy                # the lint everyone runs
cargo fmt                   # opinionated formatter, like gofmt
```

**Lockfile by default** (`Cargo.lock`). **Reproducible builds.** **Workspaces** to share deps across many crates in one repo. This is the toolchain Python's `pip + venv + pyproject + uv + poetry` ecosystem keeps trying to imitate.

## 7 · `tokio` + `axum` — async HTTP server

```rust
// src/main.rs
use axum::{routing::post, Json, Router};
use serde::{Deserialize, Serialize};
use std::net::SocketAddr;

#[derive(Deserialize)]
struct Echo { text: String }

#[derive(Serialize)]
struct Said { said: String }

async fn echo(Json(req): Json<Echo>) -> Json<Said> {
    Json(Said { said: req.text })
}

#[tokio::main]
async fn main() {
    let app = Router::new().route("/echo", post(echo));
    let addr = SocketAddr::from(([0,0,0,0], 8080));
    let listener = tokio::net::TcpListener::bind(addr).await.unwrap();
    axum::serve(listener, app).await.unwrap();
}
```

| Concept | What it gives you |
|---|---|
| **`async fn`** | a function that returns a `Future` (lazy; doesn't run until awaited) |
| **`tokio` runtime** | the executor that polls those Futures |
| **`axum`** | router + extractors (`Json`, `Path`, `Query`) — like FastAPI in Rust |
| **`serde`** | the JSON / TOML / YAML / Bincode codec, all in one trait |
| **`tower`** | middleware framework underneath axum (timeouts, retries, tracing) |

Performance: **a single Rust + axum service routinely serves 100k+ RPS on one core**. The price is paid in compile time, not runtime.

## 8 · PyO3 — call Rust from Python

The most common way to use Rust in an ML stack: write the hot loop in Rust, expose it as a Python module via **PyO3**.

```toml
# Cargo.toml
[lib]
name = "fastnorm"
crate-type = ["cdylib"]

[dependencies]
pyo3 = { version = "0.22", features = ["extension-module"] }
```

```rust
// src/lib.rs
use pyo3::prelude::*;

#[pyfunction]
fn l2_normalise(mut v: Vec<f32>) -> Vec<f32> {
    let n: f32 = v.iter().map(|x| x*x).sum::<f32>().sqrt();
    if n > 0.0 { for x in v.iter_mut() { *x /= n; } }
    v
}

#[pymodule]
fn fastnorm(_py: Python, m: &Bound<PyModule>) -> PyResult<()> {
    m.add_function(wrap_pyfunction!(l2_normalise, m)?)?;
    Ok(())
}
```

```bash
pip install maturin
maturin develop --release    # builds the .so and installs it
```

```python
import fastnorm
fastnorm.l2_normalise([3.0, 4.0])    # → [0.6, 0.8]
```

This is exactly how `tokenizers`, `safetensors`, `polars` ship a Python API — Cargo crate + thin PyO3 wrapper + maturin packaging.

## 9 · The ML-Rust ecosystem

| Crate | What it is |
|---|---|
| **`tokenizers`** (HF) | byte-pair encoding / WordPiece / SentencePiece — what every LLM uses |
| **`safetensors`** | safe alternative to PyTorch `.bin` (no pickle = no RCE) |
| **`candle`** (HF) | pure-Rust ML framework — load Llama/Mistral/Whisper without Python |
| **`burn`** | another pure-Rust framework, dynamic graphs, multi-backend |
| **`mistral.rs`** | high-perf inference of Mistral-style models in Rust |
| **`llama-rs` / `rustformers`** | community pre-cursors to candle |
| **`polars`** | DataFrame library; lazy execution, parallel by default |
| **`ndarray`** | NumPy equivalent |
| **`rayon`** | data-parallel iterators (`par_iter`) |
| **`tch`** | PyTorch C++ bindings for Rust — when you must reuse pretrained models |
| **`ort`** | ONNX Runtime bindings |
| **`qdrant`** (server) | vector DB you used in M42 |
| **`linfa`** | scikit-learn equivalent in pure Rust |
| **`nalgebra` / `glam`** | linear algebra |

A typical 2026 stack: **train in PyTorch, export to ONNX or safetensors, serve with `candle` or ORT in a Rust binary.**

## 10 · Rust vs Go vs C++

| Use case | Pick |
|---|---|
| K8s controller, HTTP gateway, sidecar | **Go** (M52) |
| Hot path inside a service: tokenisation, vector math, parsing | **Rust** |
| You need to embed in or interop with **legacy C++** | **C++** (or Rust + cxx crate) |
| GPU kernels | **CUDA C++** or **Triton**; Rust binding via `cudarc` |
| Replacing slow Python libs | **Rust + PyO3 + maturin** |
| You need every last cycle and don't care about safety | C/C++ |

**Honest take.** Most teams are right to start in Go for the platform layer — it's enough. Reach for Rust when:
1. The slow path is a Python C-extension you need to replace (`tokenizers` story).
2. You need predictable tail-latency that GC languages can't promise.
3. You're shipping the same code to **WebAssembly** in the browser, edge runtimes, and a server.

### Anti-patterns
- ❌ Picking Rust for a CRUD service "because it's fast." You'll spend three months fighting the borrow checker for a service that serves 50 RPS.
- ❌ `unsafe` blocks scattered through your code. Use them sparingly; audit them like crypto.
- ❌ Rolling your own async executor. Use `tokio`. Don't fork it.
- ❌ Manual `.unwrap()` in handlers. Crashes a worker on every malformed input. Use `?` + a real error type (`thiserror`, `anyhow`).
- ❌ Reaching for traits before you have the use-case. Generic over-design is a real smell.


## ✅ Recap
- Rust = C++ speed + memory safety + a brilliant package manager.
- The **ownership + borrowing** discipline is the price of admission. Internalise the **shared XOR mutable** rule.
- `Result<T, E>` + `?` replace exceptions; `match` is exhaustive.
- **`tokio` + `axum`** for backends; **PyO3 + maturin** to ship a Rust crate as a Python wheel.
- The ML-Rust ecosystem is real: `tokenizers`, `safetensors`, `polars`, `candle`, `burn`, `qdrant`.
- Pick Rust where you'd otherwise reach for **C++ or a Python C-extension**. Stay in Go (M52) for the rest.

Next: **M54 — TypeScript for AI Frontends** — closing the loop, the chat UI you ship to users.
